In [1]:
from spacerocks import SpaceRock
from spacerocks.time import Time
from spacerocks.observing import Observatory, Observation
from spacerocks.spice import SpiceKernel
from spacerocks.nbody import Simulation, Force
import numpy as np


import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

kernel = SpiceKernel()
kernel.load("/Users/kjnapier/data/spice/latest_leapseconds.tls")
kernel.load("/Users/kjnapier/data/spice/sb441-n16.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-1.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-2.bsp")
kernel.load("/Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc")

Loading kernel: /Users/kjnapier/data/spice/latest_leapseconds.tls
Loading kernel: /Users/kjnapier/data/spice/sb441-n16.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-1.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-2.bsp
Loading kernel: /Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc


In [2]:
w84 = Observatory.from_obscode('w84')

In [3]:
epoch = Time.now()


rock = SpaceRock.from_horizons("Arrokoth", epoch=epoch, origin="ssb", reference_plane="J2000")
sim = Simulation.horizons(epoch, "J2000", "ssb")
sim.add(rock)

In [4]:
observations = []    
for idx in range(0, 30, 5):
    sim.integrate(epoch + idx)
    observer = w84.at(epoch + idx, reference_plane="J2000", origin="ssb")
    rock = sim.get_particle("Arrokoth")
    obs = rock.observe(observer)
    observations.append(obs)

In [9]:
smear = 1/3600 * np.pi / 180

In [7]:
simulated_observations = []
for obs in observations:
    ra = obs.ra
    dec = obs.dec
    ra += np.random.normal(0, smear)
    dec += np.random.normal(0, smear)
    epoch = obs.epoch
    observer = obs.observer
    cov = [[smear**2, 0], [0, smear**2]]
    simulated_o = Observation.from_astrometry(obs.epoch, ra, dec, obs.observer)

Observation:
  ra: 5.168542814571704
  dec: -0.34107142861609857
  ra_rate: Some(0.00044573555918755775)
  dec_rate: Some(7.881918918341782e-5)
  range: Some(43.99703654521834)
  range_rate: Some(-0.006665260570229207)
  epoch: Time { epoch: 2460713.4546896406, timescale: TDB, format: JD }
  observer: Observer { spacerock: SpaceRock { name: "earth", epoch: Time { epoch: 2460713.453888889, timescale: UTC, format: JD }, reference_plane: J2000, origin: SSB, position: [[-0.7391054041952949, 0.6000706988844423, 0.26028074832708675]], velocity: [[-0.011941059645166764, -0.011655818464636529, -0.005117851433435849]], properties: Some(Properties { mass: Some(3.0034896154502038e-6), absolute_magnitude: None, gslope: None, radius: None, albedo: None }) }, observatory: GroundObservatory { obscode: "W84", lon: 5.047380146629623, lat: -0.5236462337787118, rho: 0.9995038419300848 } }


In [11]:
help(Observation.from_astrometry)

Help on built-in function from_astrometry:

from_astrometry(epoch, ra, dec, observer, covariance=None, mag=None, mag_err=None) class method of builtins.Observation

